# T37 - 同花顺Excel导入财务指标

## 4. 数据处理核心逻辑

本章详细说明财务指标的计算逻辑和数据处理核心功能。

## 4.1 环境准备

In [ ]:
# 导入标准库
import os
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from time import sleep

# 添加项目路径
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

# 导入第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
import psycopg2

# 导入项目配置
from config import (
    RISK_THRESHOLDS,
    get_postgres_engine
)

print("环境准备完成")

## 4.2 资产负债表指标计算

### 4.2.1 核心指标计算函数

In [ ]:
def calculate_balance_sheet_indicators(df):
    """
    计算资产负债表指标
    
    Parameters:
    -----------
    df : DataFrame
        包含资产负债表原始数据的DataFrame
        
    Returns:
    --------
    DataFrame : 添加了计算指标的DataFrame
    """
    result = df.copy()
    
    # 负债比率 = 总负债 / 总资产
    if 'ths_total_liab_bond' in result.columns and 'ths_total_assets_bond' in result.columns:
        result['debt_ratio'] = result['ths_total_liab_bond'] / result['ths_total_assets_bond']
    
    # 流动比率 = 流动资产 / 流动负债
    if 'ths_total_current_assets_bond' in result.columns and 'ths_total_current_liab_bond' in result.columns:
        result['current_ratio'] = result['ths_total_current_assets_bond'] / result['ths_total_current_liab_bond']
    
    # 速动比率 = (流动资产 - 存货) / 流动负债
    if all(col in result.columns for col in ['ths_total_current_assets_bond', 'ths_inventory_bond', 'ths_total_current_liab_bond']):
        result['quick_ratio'] = (result['ths_total_current_assets_bond'] - result['ths_inventory_bond']) / result['ths_total_current_liab_bond']
    
    # 资产负债率 = 总负债 / 所有者权益
    if 'ths_total_liab_bond' in result.columns and 'ths_total_owner_equity_bond' in result.columns:
        result['debt_to_equity'] = result['ths_total_liab_bond'] / result['ths_total_owner_equity_bond']
    
    # 净资产 = 总资产 - 总负债
    if 'ths_total_assets_bond' in result.columns and 'ths_total_liab_bond' in result.columns:
        result['net_assets'] = result['ths_total_assets_bond'] - result['ths_total_liab_bond']
    
    return result

# 测试资产负债表指标计算
test_balance_data = pd.DataFrame({
    'thscode': ['101660001.IB'],
    'dt': ['20230630'],
    'ths_total_assets_bond': [1000000],
    'ths_total_current_assets_bond': [400000],
    'ths_inventory_bond': [100000],
    'ths_total_liab_bond': [600000],
    'ths_total_current_liab_bond': [200000],
    'ths_total_owner_equity_bond': [400000]
})

result = calculate_balance_sheet_indicators(test_balance_data)
print("资产负债表指标计算结果:")
display(result)

### 4.2.2 指标说明

In [ ]:
# 资产负债表指标说明
balance_sheet_formulas = {
    "负债比率": {
        "公式": "总负债 / 总资产",
        "含义": "反映企业资产中由债务资金形成的比例",
        "预警线": RISK_THRESHOLDS['debt_ratio_warning'],
        "危险线": RISK_THRESHOLDS['debt_ratio_danger']
    },
    "流动比率": {
        "公式": "流动资产 / 流动负债",
        "含义": "反映企业短期偿债能力",
        "预警线": RISK_THRESHOLDS['current_ratio_warning'],
        "危险线": RISK_THRESHOLDS['current_ratio_danger']
    },
    "速动比率": {
        "公式": "(流动资产 - 存货) / 流动负债",
        "含义": "反映企业速动资产偿还流动负债的能力",
        "预警线": 1.0,
        "危险线": 0.5
    },
    "资产负债率": {
        "公式": "总负债 / 所有者权益",
        "含义": "反映企业财务杠杆程度",
        "预警线": 1.5,
        "危险线": 2.0
    }
}

print("资产负债表指标说明:")
for name, info in balance_sheet_formulas.items():
    print(f"\n{name}:")
    print(f"  公式: {info['公式']}")
    print(f"  含义: {info['含义']}")
    print(f"  预警线: {info['预警线']}")
    print(f"  危险线: {info['危险线']}")

## 4.3 利润表指标计算

### 4.3.1 核心指标计算函数

In [ ]:
def calculate_income_statement_indicators(df):
    """
    计算利润表指标
    
    Parameters:
    -----------
    df : DataFrame
        包含利润表原始数据的DataFrame
        
    Returns:
    --------
    DataFrame : 添加了计算指标的DataFrame
    """
    result = df.copy()
    
    # 注意：实际利润表数据需要从同花顺API获取
    # 这里提供计算框架
    
    # 毛利润 = 营业收入 - 营业成本
    if 'operating_revenue' in result.columns and 'operating_cost' in result.columns:
        result['gross_profit'] = result['operating_revenue'] - result['operating_cost']
    
    # 销售毛利率 = 毛利润 / 营业收入
    if 'gross_profit' in result.columns and 'operating_revenue' in result.columns:
        result['gross_margin'] = result['gross_profit'] / result['operating_revenue']
    
    # 营业利润率 = 营业利润 / 营业收入
    if 'operating_profit' in result.columns and 'operating_revenue' in result.columns:
        result['operating_margin'] = result['operating_profit'] / result['operating_revenue']
    
    # 净利率 = 净利润 / 营业收入
    if 'net_profit' in result.columns and 'operating_revenue' in result.columns:
        result['net_margin'] = result['net_profit'] / result['operating_revenue']
    
    return result

# 测试利润表指标计算
test_income_data = pd.DataFrame({
    'thscode': ['101660001.IB'],
    'dt': ['20230630'],
    'operating_revenue': [500000],
    'operating_cost': [300000],
    'operating_profit': [150000],
    'net_profit': [100000]
})

result = calculate_income_statement_indicators(test_income_data)
print("利润表指标计算结果:")
display(result)

## 4.4 现金流量表指标计算

### 4.4.1 核心指标计算函数

In [ ]:
def calculate_cash_flow_indicators(df):
    """
    计算现金流量表指标
    
    Parameters:
    -----------
    df : DataFrame
        包含现金流量表原始数据的DataFrame
        
    Returns:
    --------
    DataFrame : 添加了计算指标的DataFrame
    """
    result = df.copy()
    
    # 经营性现金流比率 = 经营活动现金流量 / 净利润
    if 'operating_cash_flow' in result.columns and 'net_profit' in result.columns:
        result['operating_cash_ratio'] = result['operating_cash_flow'] / result['net_profit']
    
    # 自由现金流 = 经营活动现金流量 - 资本支出
    if 'operating_cash_flow' in result.columns and 'capital_expenditure' in result.columns:
        result['free_cash_flow'] = result['operating_cash_flow'] - result['capital_expenditure']
    
    # 现金流覆盖率 = 经营活动现金流量 / 有息负债
    if 'operating_cash_flow' in result.columns and 'interest_bearing_debt' in result.columns:
        result['coverage_ratio'] = result['operating_cash_flow'] / result['interest_bearing_debt']
    
    return result

# 测试现金流量表指标计算
test_cash_flow_data = pd.DataFrame({
    'thscode': ['101660001.IB'],
    'dt': ['20230630'],
    'operating_cash_flow': [120000],
    'net_profit': [100000],
    'capital_expenditure': [50000],
    'interest_bearing_debt': [400000]
})

result = calculate_cash_flow_indicators(test_cash_flow_data)
print("现金流量表指标计算结果:")
display(result)

## 4.5 综合财务分析

### 4.5.1 盈利能力分析

In [ ]:
def analyze_profitability(df, balance_sheet_df=None):
    """
    分析盈利能力
    
    Parameters:
    -----------
    df : DataFrame
        利润表数据
    balance_sheet_df : DataFrame
        资产负债表数据（用于计算ROE）
        
    Returns:
    --------
    dict : 盈利能力指标
    """
    result = {}
    
    # 毛利率趋势（滚动平均）
    if 'gross_margin' in df.columns:
        result['gross_margin_trend'] = df['gross_margin'].rolling(4).mean().iloc[-1] if len(df) >= 4 else df['gross_margin'].mean()
    
    # 净利率趋势（滚动平均）
    if 'net_margin' in df.columns:
        result['net_margin_trend'] = df['net_margin'].rolling(4).mean().iloc[-1] if len(df) >= 4 else df['net_margin'].mean()
    
    # ROE（净资产收益率）= 净利润 / 所有者权益
    if balance_sheet_df is not None and 'net_profit' in df.columns and 'ths_total_owner_equity_bond' in balance_sheet_df.columns:
        result['roe'] = df['net_profit'].iloc[-1] / balance_sheet_df['ths_total_owner_equity_bond'].iloc[-1]
    
    return result

print("盈利能力分析函数定义完成")

### 4.5.2 偿债能力分析

In [ ]:
def analyze_solvency(balance_sheet_df, cash_flow_df=None, income_df=None):
    """
    分析偿债能力
    
    Parameters:
    -----------
    balance_sheet_df : DataFrame
        资产负债表数据
    cash_flow_df : DataFrame
        现金流量表数据
    income_df : DataFrame
        利润表数据
        
    Returns:
    --------
    dict : 偿债能力指标
    """
    result = {}
    
    # 流动比率
    if 'current_ratio' in balance_sheet_df.columns:
        result['current_ratio'] = balance_sheet_df['current_ratio'].iloc[-1]
    
    # 速动比率
    if 'quick_ratio' in balance_sheet_df.columns:
        result['quick_ratio'] = balance_sheet_df['quick_ratio'].iloc[-1]
    
    # 资产负债率
    if 'debt_ratio' in balance_sheet_df.columns:
        result['debt_ratio'] = balance_sheet_df['debt_ratio'].iloc[-1]
    
    # 利息保障倍数 = EBIT / 利息费用
    if income_df is not None and 'ebit' in income_df.columns and 'interest_expense' in income_df.columns:
        result['interest_coverage'] = income_df['ebit'].iloc[-1] / income_df['interest_expense'].iloc[-1]
    
    # 现金比率
    if cash_flow_df is not None and 'cash_and_equivalents' in cash_flow_df.columns and 'ths_total_current_liab_bond' in balance_sheet_df.columns:
        result['cash_ratio'] = cash_flow_df['cash_and_equivalents'].iloc[-1] / balance_sheet_df['ths_total_current_liab_bond'].iloc[-1]
    
    return result

print("偿债能力分析函数定义完成")

### 4.5.3 运营效率分析

In [ ]:
def analyze_operational_efficiency(income_df, balance_sheet_df):
    """
    分析运营效率
    
    Parameters:
    -----------
    income_df : DataFrame
        利润表数据
    balance_sheet_df : DataFrame
        资产负债表数据
        
    Returns:
    --------
    dict : 运营效率指标
    """
    result = {}
    
    # 存货周转率 = 营业成本 / 存货
    if 'operating_cost' in income_df.columns and 'ths_inventory_bond' in balance_sheet_df.columns:
        result['inventory_turnover'] = income_df['operating_cost'].iloc[-1] / balance_sheet_df['ths_inventory_bond'].iloc[-1]
    
    # 应收账款周转率 = 营业收入 / 应收账款
    if 'operating_revenue' in income_df.columns and 'ths_account_receivable_bond' in balance_sheet_df.columns:
        result['receivables_turnover'] = income_df['operating_revenue'].iloc[-1] / balance_sheet_df['ths_account_receivable_bond'].iloc[-1]
    
    # 总资产周转率 = 营业收入 / 总资产
    if 'operating_revenue' in income_df.columns and 'ths_total_assets_bond' in balance_sheet_df.columns:
        result['asset_turnover'] = income_df['operating_revenue'].iloc[-1] / balance_sheet_df['ths_total_assets_bond'].iloc[-1]
    
    return result

print("运营效率分析函数定义完成")

## 4.6 综合财务评分

In [ ]:
def calculate_financial_score(balance_sheet_df, income_df=None, cash_flow_df=None):
    """
    计算综合财务评分
    
    Parameters:
    -----------
    balance_sheet_df : DataFrame
        资产负债表数据
    income_df : DataFrame
        利润表数据
    cash_flow_df : DataFrame
        现金流量表数据
        
    Returns:
    --------
    dict : 综合评分和各维度得分
    """
    scores = {}
    
    # 1. 偿债能力评分 (40分)
    solvency_score = 0
    if 'debt_ratio' in balance_sheet_df.columns:
        debt_ratio = balance_sheet_df['debt_ratio'].iloc[-1]
        if debt_ratio < 0.4:
            solvency_score += 20
        elif debt_ratio < 0.6:
            solvency_score += 15
        elif debt_ratio < 0.8:
            solvency_score += 10
        else:
            solvency_score += 5
    
    if 'current_ratio' in balance_sheet_df.columns:
        current_ratio = balance_sheet_df['current_ratio'].iloc[-1]
        if current_ratio > 2.0:
            solvency_score += 20
        elif current_ratio > 1.5:
            solvency_score += 15
        elif current_ratio > 1.0:
            solvency_score += 10
        else:
            solvency_score += 5
    
    scores['solvency_score'] = solvency_score
    
    # 2. 盈利能力评分 (30分)
    profitability_score = 0
    if income_df is not None and 'net_margin' in income_df.columns:
        net_margin = income_df['net_margin'].iloc[-1]
        if net_margin > 0.15:
            profitability_score += 15
        elif net_margin > 0.10:
            profitability_score += 12
        elif net_margin > 0.05:
            profitability_score += 8
        else:
            profitability_score += 5
    
    if income_df is not None and 'gross_margin' in income_df.columns:
        gross_margin = income_df['gross_margin'].iloc[-1]
        if gross_margin > 0.30:
            profitability_score += 15
        elif gross_margin > 0.20:
            profitability_score += 12
        elif gross_margin > 0.10:
            profitability_score += 8
        else:
            profitability_score += 5
    
    scores['profitability_score'] = profitability_score
    
    # 3. 运营效率评分 (30分)
    efficiency_score = 15  # 基础分
    
    if income_df is not None and 'operating_revenue' in income_df.columns:
        # 简化：基于收入规模给分
        revenue = income_df['operating_revenue'].iloc[-1]
        if revenue > 1000000:  # 100万以上
            efficiency_score += 15
        elif revenue > 500000:
            efficiency_score += 10
        else:
            efficiency_score += 5
    
    scores['efficiency_score'] = efficiency_score
    
    # 总分
    scores['total_score'] = solvency_score + profitability_score + efficiency_score
    
    # 评级
    total = scores['total_score']
    if total >= 80:
        scores['rating'] = '优秀'
    elif total >= 60:
        scores['rating'] = '良好'
    elif total >= 40:
        scores['rating'] = '一般'
    else:
        scores['rating'] = '较差'
    
    return scores

# 测试综合评分
test_balance = calculate_balance_sheet_indicators(test_balance_data)
test_income = calculate_income_statement_indicators(test_income_data)

scores = calculate_financial_score(test_balance, test_income)
print("综合财务评分:")
for key, value in scores.items():
    print(f"  {key}: {value}")

## 4.7 数据验证和清洗

In [ ]:
def validate_financial_data(df):
    """
    验证财务数据有效性
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
        
    Returns:
    --------
    dict : 验证结果
    """
    result = {
        'is_valid': True,
        'errors': [],
        'warnings': []
    }
    
    # 检查空值
    null_counts = df.isnull().sum()
    for col, count in null_counts.items():
        if count > 0:
            result['warnings'].append(f"列 {col} 有 {count} 个空值")
    
    # 检查负值（对于不应该为负的指标）
    non_negative_cols = ['ths_total_assets_bond', 'ths_currency_fund_bond', 'ths_inventory_bond']
    for col in non_negative_cols:
        if col in df.columns:
            negative_count = (df[col] < 0).sum()
            if negative_count > 0:
                result['errors'].append(f"列 {col} 有 {negative_count} 个负值")
                result['is_valid'] = False
    
    # 检查负债比率范围
    if 'debt_ratio' in df.columns:
        invalid_ratios = ((df['debt_ratio'] < 0) | (df['debt_ratio'] > 1)).sum()
        if invalid_ratios > 0:
            result['warnings'].append(f"有 {invalid_ratios} 个负债比率不在0-1范围内")
    
    return result

def clean_financial_data(df):
    """
    清洗财务数据
    
    Parameters:
    -----------
    df : DataFrame
        原始财务数据
        
    Returns:
    --------
    DataFrame : 清洗后的数据
    """
    result = df.copy()
    
    # 删除全为空的行
    result.dropna(how='all', inplace=True)
    
    # 用中位数填充数值列的空值
    numeric_cols = result.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if result[col].isnull().any():
            median_value = result[col].median()
            result[col].fillna(median_value, inplace=True)
    
    return result

print("数据验证和清洗函数定义完成")